# MIMIC-IV demo — calibration go/no-go check

`docs/architecture.md` §1.1 proposed using the MIMIC-IV Clinical Database Demo to calibrate two
simulator parameters: **specimen-type mix** and the **collection time-of-day profile**. This notebook
tests whether the data can actually support that, on two criteria per parameter:

1. **Sample size** — are there enough specimens per cell to estimate anything?
2. **Population** — is this the right kind of laboratory? Tracer's target setting is a general clinical
   microbiology laboratory serving inpatient, emergency and **outpatient** collection. MIMIC-IV is Beth
   Israel Deaconess critical care data.

A finding of *not usable* is a legitimate result. Per §1.1's fallback rule, a parameter that fails either
criterion reverts to its assumed value rather than being fitted on thin or mismatched data.

**Source.** MIMIC-IV Clinical Database Demo v2.2, PhysioNet, Open Data Commons Open Database License.
Not redistributed in this repository; `data/` is gitignored.

No calibration code is written here. This notebook only decides whether to write any.

In [1]:
import os
import duckdb

ROOT = ".." if os.path.basename(os.getcwd()) == "notebooks" else "."
BASE = f"{ROOT}/data/mimic-demo/physionet.org/files/mimic-iv-demo/2.2"
MICRO = f"{BASE}/hosp/microbiologyevents.csv.gz"

con = duckdb.connect()

# One row per SPECIMEN, not per row of the table. microbiologyevents carries one row per
# organism-antibiotic result, so a single specimen appears many times; counting rows would
# overstate the sample by roughly a factor of two and weight the mix toward whatever grew.
con.sql(f"""
CREATE OR REPLACE VIEW spec AS
SELECT micro_specimen_id,
       any_value(subject_id)     AS subject_id,
       any_value(hadm_id)        AS hadm_id,
       any_value(spec_type_desc) AS spec_type_desc,
       min(charttime)            AS charttime,
       min(chartdate)            AS chartdate,
       max(storetime)            AS storetime
FROM read_csv_auto('{MICRO}')
GROUP BY micro_specimen_id
""")
print("connected")

connected


## 1. Volume

In [2]:
con.sql(f"""
SELECT count(*)                          AS rows,
       count(DISTINCT micro_specimen_id) AS specimens,
       count(DISTINCT subject_id)        AS subjects,
       count(DISTINCT hadm_id)           AS admissions,
       min(chartdate)                    AS first_chartdate,
       max(chartdate)                    AS last_chartdate,
       count(DISTINCT spec_type_desc)    AS spec_types
FROM read_csv_auto('{MICRO}')
""").show()

┌───────┬───────────┬──────────┬────────────┬─────────────────────┬─────────────────────┬────────────┐
│ rows  │ specimens │ subjects │ admissions │   first_chartdate   │   last_chartdate    │ spec_types │
│ int64 │   int64   │  int64   │   int64    │      timestamp      │      timestamp      │   int64    │
├───────┼───────────┼──────────┼────────────┼─────────────────────┼─────────────────────┼────────────┤
│  2899 │      1336 │       97 │        171 │ 2110-01-04 00:00:00 │ 2202-01-10 00:00:00 │         35 │
└───────┴───────────┴──────────┴────────────┴─────────────────────┴─────────────────────┴────────────┘



In [3]:
# Null rates on the two timestamp columns the whole question depends on.
con.sql(f"""
SELECT count(*) AS rows,
       sum(CASE WHEN charttime IS NULL THEN 1 ELSE 0 END) AS charttime_null,
       sum(CASE WHEN storetime IS NULL THEN 1 ELSE 0 END) AS storetime_null,
       sum(CASE WHEN chartdate IS NULL THEN 1 ELSE 0 END) AS chartdate_null
FROM read_csv_auto('{MICRO}')
""").show()

con.sql("""
SELECT CASE WHEN charttime IS NULL THEN 'charttime NULL (date only)' ELSE 'charttime present' END AS status,
       count(*) AS specimens,
       round(100.0*count(*)/sum(count(*)) OVER (), 1) AS pct
FROM spec GROUP BY 1
""").show()

┌───────┬────────────────┬────────────────┬────────────────┐
│ rows  │ charttime_null │ storetime_null │ chartdate_null │
│ int64 │     int128     │     int128     │     int128     │
├───────┼────────────────┼────────────────┼────────────────┤
│  2899 │            183 │             16 │              0 │
└───────┴────────────────┴────────────────┴────────────────┘

┌────────────────────────────┬───────────┬────────┐
│           status           │ specimens │  pct   │
│          varchar           │   int64   │ double │
├────────────────────────────┼───────────┼────────┤
│ charttime present          │      1291 │   96.6 │
│ charttime NULL (date only) │        45 │    3.4 │
└────────────────────────────┴───────────┴────────┘



## 2. Specimen mix

In [4]:
con.sql("""
SELECT spec_type_desc,
       count(*) AS specimens,
       round(100.0*count(*)/sum(count(*)) OVER (), 2) AS pct,
       count(DISTINCT subject_id) AS patients
FROM spec GROUP BY 1 ORDER BY specimens DESC
""").show(max_rows=40)

┌──────────────────────────────────────────┬───────────┬────────┬──────────┐
│              spec_type_desc              │ specimens │  pct   │ patients │
│                 varchar                  │   int64   │ double │  int64   │
├──────────────────────────────────────────┼───────────┼────────┼──────────┤
│ BLOOD CULTURE                            │       492 │  36.83 │       60 │
│ URINE                                    │       275 │  20.58 │       77 │
│ MRSA SCREEN                              │       102 │   7.63 │       78 │
│ STOOL                                    │        65 │   4.87 │       28 │
│ SEROLOGY/BLOOD                           │        53 │   3.97 │       19 │
│ SPUTUM                                   │        51 │   3.82 │       32 │
│ Immunology (CMV)                         │        41 │   3.07 │        4 │
│ SWAB                                     │        40 │   2.99 │       19 │
│ PERITONEAL FLUID                         │        25 │   1.87 │        7 │

In [5]:
con.sql("""
WITH c AS (SELECT spec_type_desc, count(*) n FROM spec GROUP BY 1)
SELECT count(*)                                AS types_total,
       sum(CASE WHEN n < 30 THEN 1 ELSE 0 END) AS types_under_30,
       sum(CASE WHEN n < 10 THEN 1 ELSE 0 END) AS types_under_10,
       sum(CASE WHEN n = 1  THEN 1 ELSE 0 END) AS types_with_exactly_1
FROM c
""").show()

┌─────────────┬────────────────┬────────────────┬──────────────────────┐
│ types_total │ types_under_30 │ types_under_10 │ types_with_exactly_1 │
│    int64    │     int128     │     int128     │        int128        │
├─────────────┼────────────────┼────────────────┼──────────────────────┤
│          35 │             27 │             18 │                    3 │
└─────────────┴────────────────┴────────────────┴──────────────────────┘



In [6]:
# The question that actually matters: how does this map onto the five types the prototype models?
con.sql("""
SELECT CASE
  WHEN spec_type_desc = 'CSF;SPINAL FLUID' THEN '1 CSF'
  WHEN spec_type_desc = 'URINE' THEN '2/3 Urine (container unknown)'
  WHEN spec_type_desc IN ('SWAB','ABSCESS','Staph aureus swab','FOOT CULTURE','EAR') THEN '4 Wound/swab (approx)'
  WHEN spec_type_desc IN ('SPUTUM','BRONCHOALVEOLAR LAVAGE','BRONCHIAL WASHINGS','Mini-BAL','BRONCHIAL BRUSH')
       THEN '5 Lower respiratory (approx)'
  WHEN spec_type_desc LIKE '%BLOOD CULTURE%' OR spec_type_desc = 'Stem Cell - Blood Culture'
       THEN 'X Blood culture (EXCLUDED from the prototype, arch 6.1)'
  ELSE 'Z Not modelled' END AS modelled_as,
  count(*) AS specimens,
  round(100.0*count(*)/sum(count(*)) OVER (), 1) AS pct
FROM spec GROUP BY 1 ORDER BY 1
""").show()

┌─────────────────────────────────────────────────────────┬───────────┬────────┐
│                       modelled_as                       │ specimens │  pct   │
│                         varchar                         │   int64   │ double │
├─────────────────────────────────────────────────────────┼───────────┼────────┤
│ 1 CSF                                                   │         8 │    0.6 │
│ 2/3 Urine (container unknown)                           │       275 │   20.6 │
│ 4 Wound/swab (approx)                                   │        74 │    5.5 │
│ 5 Lower respiratory (approx)                            │        81 │    6.1 │
│ X Blood culture (EXCLUDED from the prototype, arch 6.1) │       512 │   38.3 │
│ Z Not modelled                                          │       386 │   28.9 │
└─────────────────────────────────────────────────────────┴───────────┴────────┘



## 3. Collection time of day

In [7]:
con.sql("""
SELECT extract('hour' FROM charttime) AS hour, count(*) AS specimens
FROM spec WHERE charttime IS NOT NULL GROUP BY 1 ORDER BY 1
""").show(max_rows=26)

con.sql("""
WITH h AS (SELECT extract('hour' FROM charttime) hr, count(*) n
           FROM spec WHERE charttime IS NOT NULL GROUP BY 1)
SELECT count(*) AS hours_present, 24 - count(*) AS hours_absent,
       sum(CASE WHEN n < 10 THEN 1 ELSE 0 END) AS hours_under_10,
       sum(CASE WHEN n < 30 THEN 1 ELSE 0 END) AS hours_under_30,
       min(n) AS min_n, max(n) AS max_n, round(avg(n),1) AS mean_n
FROM h
""").show()

┌───────┬───────────┐
│ hour  │ specimens │
│ int64 │   int64   │
├───────┼───────────┤
│     0 │        26 │
│     1 │        25 │
│     2 │        44 │
│     3 │        26 │
│     4 │        45 │
│     5 │        42 │
│     6 │        45 │
│     7 │        20 │
│     8 │        44 │
│     9 │        75 │
│    10 │        75 │
│    11 │        80 │
│    12 │        77 │
│    13 │        89 │
│    14 │        71 │
│    15 │        68 │
│    16 │        80 │
│    17 │        62 │
│    18 │        51 │
│    19 │        47 │
│    20 │        44 │
│    21 │        68 │
│    22 │        51 │
│    23 │        36 │
└───────┴───────────┘
       24 rows     



┌───────────────┬──────────────┬────────────────┬────────────────┬───────┬───────┬────────┐
│ hours_present │ hours_absent │ hours_under_10 │ hours_under_30 │ min_n │ max_n │ mean_n │
│     int64     │    int64     │     int128     │     int128     │ int64 │ int64 │ double │
├───────────────┼──────────────┼────────────────┼────────────────┼───────┼───────┼────────┤
│            24 │            0 │              0 │              4 │    20 │    89 │   53.8 │
└───────────────┴──────────────┴────────────────┴────────────────┴───────┴───────┴────────┘



In [8]:
# Shape, not just count. An acute-care ward draws overnight; an outpatient-serving lab does not.
con.sql("""
SELECT CASE WHEN extract('hour' FROM charttime) BETWEEN 0 AND 6  THEN '00-06 overnight'
            WHEN extract('hour' FROM charttime) BETWEEN 7 AND 17 THEN '07-17 day'
            ELSE '18-23 evening' END AS band,
       count(*) AS specimens, round(100.0*count(*)/sum(count(*)) OVER (),1) AS pct
FROM spec WHERE charttime IS NOT NULL GROUP BY 1 ORDER BY 1
""").show()

con.sql("""
SELECT extract('hour' FROM charttime) AS hour,
       count(*) FILTER (WHERE spec_type_desc = 'BLOOD CULTURE')  AS blood_culture,
       count(*) FILTER (WHERE spec_type_desc <> 'BLOOD CULTURE') AS everything_else
FROM spec WHERE charttime IS NOT NULL GROUP BY 1 ORDER BY 1
""").show(max_rows=26)

┌─────────────────┬───────────┬────────┐
│      band       │ specimens │  pct   │
│     varchar     │   int64   │ double │
├─────────────────┼───────────┼────────┤
│ 00-06 overnight │       253 │   19.6 │
│ 07-17 day       │       741 │   57.4 │
│ 18-23 evening   │       297 │   23.0 │
└─────────────────┴───────────┴────────┘



┌───────┬───────────────┬─────────────────┐
│ hour  │ blood_culture │ everything_else │
│ int64 │     int64     │      int64      │
├───────┼───────────────┼─────────────────┤
│     0 │            10 │              16 │
│     1 │            11 │              14 │
│     2 │            21 │              23 │
│     3 │            10 │              16 │
│     4 │            23 │              22 │
│     5 │            23 │              19 │
│     6 │            34 │              11 │
│     7 │            10 │              10 │
│     8 │            16 │              28 │
│     9 │            21 │              54 │
│    10 │            26 │              49 │
│    11 │            18 │              62 │
│    12 │            22 │              55 │
│    13 │            28 │              61 │
│    14 │            22 │              49 │
│    15 │            18 │              50 │
│    16 │            24 │              56 │
│    17 │            24 │              38 │
│    18 │            23 │       

## 4. Collection-to-result envelope

`storetime` is when the final result was documented. For culture this includes incubation, so this
interval is expected to be days rather than minutes — the check is how far from a transport interval it is.

In [9]:
con.sql("""
WITH e AS (SELECT date_diff('minute', charttime, storetime)/60.0 AS hrs
           FROM spec WHERE charttime IS NOT NULL AND storetime IS NOT NULL)
SELECT count(*) AS n,
       round(min(hrs),2) AS min_h, round(quantile_cont(hrs,0.25),2) AS p25_h,
       round(median(hrs),2) AS median_h, round(quantile_cont(hrs,0.75),2) AS p75_h,
       round(quantile_cont(hrs,0.90),2) AS p90_h, round(max(hrs),2) AS max_h
FROM e
""").show()

con.sql("""
SELECT round(100.0*sum(CASE WHEN date_diff('minute',charttime,storetime) <= 120  THEN 1 ELSE 0 END)/count(*),2) AS pct_under_2h,
       round(100.0*sum(CASE WHEN date_diff('minute',charttime,storetime) <= 480  THEN 1 ELSE 0 END)/count(*),2) AS pct_under_8h,
       round(100.0*sum(CASE WHEN date_diff('minute',charttime,storetime) <= 1440 THEN 1 ELSE 0 END)/count(*),2) AS pct_under_24h
FROM spec WHERE charttime IS NOT NULL AND storetime IS NOT NULL
""").show()

┌───────┬────────┬────────┬──────────┬────────┬────────┬─────────┐
│   n   │ min_h  │ p25_h  │ median_h │ p75_h  │ p90_h  │  max_h  │
│ int64 │ double │ double │  double  │ double │ double │ double  │
├───────┼────────┼────────┼──────────┼────────┼────────┼─────────┤
│  1291 │   1.55 │   37.3 │    80.07 │ 140.64 │ 149.58 │ 1816.98 │
└───────┴────────┴────────┴──────────┴────────┴────────┴─────────┘



┌──────────────┬──────────────┬───────────────┐
│ pct_under_2h │ pct_under_8h │ pct_under_24h │
│    double    │    double    │    double     │
├──────────────┼──────────────┼───────────────┤
│         0.15 │         3.49 │         10.15 │
└──────────────┴──────────────┴───────────────┘



In [10]:
con.sql("""
WITH e AS (SELECT spec_type_desc, date_diff('minute', charttime, storetime)/60.0 AS hrs
           FROM spec WHERE charttime IS NOT NULL AND storetime IS NOT NULL)
SELECT spec_type_desc, count(*) AS n,
       round(median(hrs),2) AS median_h,
       round(quantile_cont(hrs,0.90),2) AS p90_h,
       round(max(hrs),2) AS max_h,
       round(100.0*sum(CASE WHEN hrs < 4 THEN 1 ELSE 0 END)/count(*),1) AS pct_under_4h
FROM e GROUP BY 1 HAVING count(*) >= 5 ORDER BY n DESC
""").show(max_rows=30)

┌──────────────────────────────────────────┬───────┬──────────┬─────────┬─────────┬──────────────┐
│              spec_type_desc              │   n   │ median_h │  p90_h  │  max_h  │ pct_under_4h │
│                 varchar                  │ int64 │  double  │ double  │ double  │    double    │
├──────────────────────────────────────────┼───────┼──────────┼─────────┼─────────┼──────────────┤
│ BLOOD CULTURE                            │   488 │   139.68 │  149.88 │  182.28 │          0.0 │
│ URINE                                    │   256 │    31.87 │   61.39 │  360.85 │          0.0 │
│ MRSA SCREEN                              │   101 │    56.72 │   64.63 │   94.35 │          0.0 │
│ STOOL                                    │    64 │    18.52 │   57.29 │  153.52 │          3.1 │
│ SPUTUM                                   │    51 │    61.55 │  167.08 │ 1576.43 │          5.9 │
│ SEROLOGY/BLOOD                           │    49 │    25.47 │   72.53 │  123.58 │          2.0 │
│ Immunolo

## 5. Population

Sample size is only half the question. This section asks what kind of laboratory produced these specimens.

In [11]:
con.sql(f"""
SELECT count(DISTINCT subject_id) AS subjects_with_an_icu_stay
FROM read_csv_auto('{BASE}/icu/icustays.csv.gz')
""").show()

con.sql(f"""
SELECT admission_location, count(*) AS admissions,
       round(100.0*count(*)/sum(count(*)) OVER (),1) AS pct
FROM read_csv_auto('{BASE}/hosp/admissions.csv.gz') GROUP BY 1 ORDER BY 2 DESC
""").show(max_rows=12)

┌───────────────────────────┐
│ subjects_with_an_icu_stay │
│           int64           │
├───────────────────────────┤
│                       100 │
└───────────────────────────┘

┌────────────────────────────────────────┬────────────┬────────┐
│           admission_location           │ admissions │  pct   │
│                varchar                 │   int64    │ double │
├────────────────────────────────────────┼────────────┼────────┤
│ EMERGENCY ROOM                         │        134 │   48.7 │
│ PHYSICIAN REFERRAL                     │         63 │   22.9 │
│ TRANSFER FROM HOSPITAL                 │         43 │   15.6 │
│ CLINIC REFERRAL                        │         10 │    3.6 │
│ PROCEDURE SITE                         │          7 │    2.5 │
│ WALK-IN/SELF REFERRAL                  │          7 │    2.5 │
│ TRANSFER FROM SKILLED NURSING FACILITY │          4 │    1.5 │
│ PACU                                   │          4 │    1.5 │
│ INTERNAL TRANSFER TO OR FROM PSYCH   

## 6. Verdict

**Specimen mix — NO.** Fails both criteria. 27 of 35 types have fewer than 30 specimens and 18 have fewer
than 10. More decisively, blood culture is 38.3% of all specimens and the prototype excludes blood cultures
(§6.1), while CSF — the highest-urgency type Tracer models — has 8. Urine cannot be split into preserved
and unpreserved, which is the contrast the policy table exists to express.

**Collection time-of-day — NO.** Passes sample size (1,291 specimens, all 24 hours populated, minimum 20
per hour) and fails population. 19.6% of collections fall between midnight and 06:00, and the single
largest blood-culture hour is 06:00 — the signature of inpatient pre-dawn draw rounds. A general laboratory
serving outpatient collection has a morning outpatient peak and a near-empty night. Importing this curve
would make the simulator more specifically an ICU, not more realistic.

**Both parameters revert to their assumed values.** No calibration code is written.

**Would credentialed access change this?** No. It would fix sample size and nothing else: MIMIC-IV is the
same hospital and the same critical-care population, so the mix stays blood-culture-dominated and the hour
curve keeps its inpatient shape. The obstacle is the population, and more rows of the same population do
not move it.

Written up in `docs/mimic-demo-findings.md`.